# RQ1 - Effectiveness and Baselines

This notebook rebuilds the RQ1 tables and figures from released CSV files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
RQ1 = RESULTS / 'rq1_effectiveness_baselines'

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PALETTE = {
    'MetaMatch': '#1F77B4',
    'LLMATCH': '#D62728',
    'SMUTF': '#9467BD',
    'MagnetoFTGPT': '#FF7F0E',
    'MagnetoFT': '#FDB462',
    'MagnetoGPT': '#E377C2',
    'Magneto': '#8C564B',
    'ISResMat': '#17BECF',
    'COMA++': '#7F7F7F',
    'COMA': '#BCBD22',
    'Similarity Flooding': '#2CA02C',
    'Distribution Based': '#AEC7E8',
    'Cupid': '#C5B0D5',
}
ORDER = list(PALETTE)

def save_fig(fig, name):
    FIGURES.mkdir(exist_ok=True)
    fig.savefig(FIGURES / f'{name}.png', bbox_inches='tight')
    fig.savefig(FIGURES / f'{name}.pdf', bbox_inches='tight')

## MetaMatch Classifier Comparison

In [ ]:
classifiers = pd.read_csv(RQ1 / 'table_1_metamatch_classifiers_all_to_all.csv')
cols = [
    'classifier',
    'mean_f1_all_to_all', 'std_f1_all_to_all',
    'mean_precision_all_to_all', 'std_precision_all_to_all',
    'mean_recall_all_to_all', 'std_recall_all_to_all',
    'n_pair_id',
]
display(classifiers[cols].round(2))

In [ ]:
latex_table = classifiers.assign(
    F1=classifiers['mean_f1_all_to_all'].map('{:.2f}'.format) + ' $\\pm$ ' + classifiers['std_f1_all_to_all'].map('{:.2f}'.format),
    Precision=classifiers['mean_precision_all_to_all'].map('{:.2f}'.format) + ' $\\pm$ ' + classifiers['std_precision_all_to_all'].map('{:.2f}'.format),
    Recall=classifiers['mean_recall_all_to_all'].map('{:.2f}'.format) + ' $\\pm$ ' + classifiers['std_recall_all_to_all'].map('{:.2f}'.format),
)[['classifier', 'F1', 'Precision', 'Recall']]
print(latex_table.to_latex(index=False, escape=False))

## Baseline Summary Table

In [ ]:
baselines = pd.read_csv(RQ1 / 'table_1_effectiveness_baselines_summary.csv')
baselines = baselines[[m in ORDER for m in baselines['method']]].copy()
baselines['method'] = pd.Categorical(baselines['method'], ORDER, ordered=True)
baselines = baselines.sort_values('method')
display(baselines.round(2))

## F1 Distribution by Table Pair

In [ ]:
pair_scores = pd.read_csv(RQ1 / 'figure_2_effectiveness_baselines_metamatch_by_pair.csv')
pair_scores = pair_scores[pair_scores['method'].isin(ORDER)].copy()
pair_scores['method'] = pd.Categorical(pair_scores['method'], ORDER, ordered=True)
pair_scores = pair_scores.sort_values('method')

fig, ax = plt.subplots(figsize=(10, 4.8))
positions = np.arange(len(ORDER))
data = [pair_scores.loc[pair_scores['method'] == m, 'f1_all_to_all'].dropna().to_numpy() for m in ORDER]
parts = ax.violinplot(data, positions=positions, showmeans=False, showmedians=True, widths=0.8)
for body, method in zip(parts['bodies'], ORDER):
    body.set_facecolor(PALETTE[method])
    body.set_edgecolor('#222222')
    body.set_alpha(0.75)
for key in ['cmins', 'cmaxes', 'cbars', 'cmedians']:
    parts[key].set_color('#222222')
    parts[key].set_linewidth(1.0)
ax.set_xticks(positions)
ax.set_xticklabels(ORDER, rotation=45, ha='right')
ax.set_ylabel('F1')
ax.set_ylim(-0.02, 1.02)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
save_fig(fig, 'plot_distribution_baseline')
plt.show()

## Pairwise Winrate Heatmap

In [ ]:
win = pd.read_csv(RQ1 / 'figure_2_winrate_f1_all_to_all_no_ties_percent_matrix.csv', index_col=0)
methods = [m for m in ORDER if m in win.index and m in win.columns]
win = win.loc[methods, methods].astype(float).round(2)
np.fill_diagonal(win.values, np.nan)

cmap = LinearSegmentedColormap.from_list('red_white_green', ['#C92A2A', '#FFFFFF', '#2B8A3E'])
masked = np.ma.masked_invalid(win.to_numpy())
fig, ax = plt.subplots(figsize=(10.5, 8.5))
im = ax.imshow(masked, cmap=cmap, vmin=0, vmax=100)
ax.set_xticks(np.arange(len(methods)))
ax.set_yticks(np.arange(len(methods)))
ax.set_xticklabels(methods, rotation=45, ha='right')
ax.set_yticklabels(methods)
for i in range(len(methods)):
    for j in range(len(methods)):
        value = win.iloc[i, j]
        label = '-' if pd.isna(value) else f'{value:.2f}'
        color = 'white' if not pd.isna(value) and (value < 30 or value > 70) else '#1f2937'
        ax.text(j, i, label, ha='center', va='center', fontsize=7.5, color=color)
fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
fig.tight_layout()
save_fig(fig, 'plot_winrate_f1_all_to_all_no_ties_matplotlib_percent_heatmap')
plt.show()

## Effectiveness by Relation and Dataset

In [ ]:
group_scores = pd.read_csv(RQ1 / 'figure_rq1_effectiveness_by_group_pair_values.csv')
group_scores = group_scores[group_scores['method'].isin(ORDER)].copy()
group_scores['method'] = pd.Categorical(group_scores['method'], ORDER, ordered=True)

def grouped_boxplot(df, group_col, filename):
    groups = sorted(df[group_col].dropna().unique())
    fig, axes = plt.subplots(len(groups), 1, figsize=(10, 2.5 * len(groups)), sharex=True)
    if len(groups) == 1:
        axes = [axes]
    for ax, group in zip(axes, groups):
        sub = df[df[group_col] == group]
        present = [m for m in ORDER if m in set(sub['method'])]
        data = [sub.loc[sub['method'] == m, 'f1_all_to_all'].dropna().to_numpy() for m in present]
        bp = ax.boxplot(data, patch_artist=True, widths=0.65, showfliers=False)
        for patch, method in zip(bp['boxes'], present):
            patch.set_facecolor(PALETTE[method])
            patch.set_alpha(0.75)
        ax.set_ylabel(str(group))
        ax.set_ylim(-0.02, 1.02)
        ax.grid(axis='y', alpha=0.25)
    axes[-1].set_xticks(range(1, len(present) + 1))
    axes[-1].set_xticklabels(present, rotation=45, ha='right')
    fig.tight_layout()
    save_fig(fig, filename)
    plt.show()

grouped_boxplot(group_scores, 'relation_group', 'plot_effectiveness_by_relation_distribution')
grouped_boxplot(group_scores, 'dataset_group', 'plot_effectiveness_by_dataset_distribution')